# copilot

> Run GitHub Copilot models through the `Chat` API with a Copilot subscription.

Copilot provides an OpenAI-compatible chat endpoint. Each request needs a short-lived Copilot token and headers that identify an editor. This module adds Copilot sign-in to `RemoteChat`. Copilot models use the same interface as other backends.

GitHub does not document or support this use. The module copies the requests that editors send. The Copilot subscription terms apply.

In [ ]:
#| default_exp copilot

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio, json, os, time
import httpx
from dataclasses import dataclass, replace
from fastcore.all import L, Path, store_attr, first
import rishi.core
from urai import Chat, ChatOpts, ToolCall, display_stream, get_runtime, resolve_runtime, resp_text, is_media
from rishi.remote import RemoteChat

In [ ]:
from fastcore.test import test_eq, test_fail

## The wire

Copilot provides OpenAI chat completions at `{endpoint}/chat/completions`. The token exchange returns the endpoint for the account plan. `COPILOT_API` is the fallback when the response has no endpoint.

Copilot requires headers from a recognised editor:

- `copilot-integration-id` identifies the editor.
- The version headers match the VS Code chat extension.
- `x-initiator` distinguishes user turns from tool-loop turns.
- `copilot-vision-request` enables image input.

Editors also send `x-request-id`. rishi omits it. fastllm caches HTTP clients by constructor arguments. A changing request id would create a new client for each request.

In [ ]:
#| export
#: Copilot's endpoint when the token exchange returns no endpoint.
COPILOT_API = 'https://api.githubcopilot.com'
#: URL that exchanges a GitHub OAuth token for a short-lived Copilot token.
COPILOT_TOKEN_URL = 'https://api.github.com/copilot_internal/v2/token'
#: OAuth app used by editor sign-in. GitHub issues Copilot tokens only for this client id.
COPILOT_CLIENT_ID = 'Iv1.b507a08c87ecfe98'

#: Editor identity sent to Copilot. Rebind these values to use another identity.
EDITOR_VERSION = 'vscode/1.99.3'
PLUGIN_VERSION = 'copilot-chat/0.26.7'
USER_AGENT     = 'GitHubCopilotChat/0.26.7'
INTEGRATION_ID = 'vscode-chat'
API_VERSION    = '2025-04-01'

def copilot_hdrs(auth=None, *, integration_id=INTEGRATION_ID, vision=False, initiator=None,
                 intent='conversation-panel'):
    "Build the headers Copilot requires. Use `auth` as the complete `Authorization` value when it is given."
    h = {'editor-version': EDITOR_VERSION, 'editor-plugin-version': PLUGIN_VERSION,
         'user-agent': USER_AGENT, 'copilot-integration-id': integration_id,
         'x-github-api-version': API_VERSION}
    if intent: h['openai-intent'] = intent
    if auth: h['Authorization'] = auth
    if vision: h['copilot-vision-request'] = 'true'
    if initiator: h['x-initiator'] = initiator
    return h

In [ ]:
h = copilot_hdrs()
test_eq(h['copilot-integration-id'], 'vscode-chat')
assert 'Authorization' not in h                      # fastllm sets it from the api key
assert 'x-request-id' not in h                       # would defeat fastllm's client cache
test_eq(copilot_hdrs('Bearer t')['Authorization'], 'Bearer t')
test_eq(copilot_hdrs(vision=True)['copilot-vision-request'], 'true')
test_eq(copilot_hdrs(initiator='agent')['x-initiator'], 'agent')

## The GitHub token

Copilot requires a GitHub OAuth token issued to a recognised application. Personal access tokens do not work. Editor sign-ins store a usable token in a configuration file. `copilot_login()` stores the same token shape in `copilot.json` under `~/.config/rishi`. `XDG_CONFIG_HOME` replaces `~/.config` when it is set.

`copilot_oauth()` looks for a token in this order:

1. The `token` argument.
2. The environment variables in `COPILOT_ENVS`.
3. The files that `oauth_paths()` lists, with rishi's own file first, then editor files.
4. The environment variables in `GITHUB_ENVS`, `GH_TOKEN` and `GITHUB_TOKEN`. These variables come last because they often hold personal access tokens.

The file reader finds an `oauth_token` key at any nesting depth up to 4.

In [ ]:
#| export
COPILOT_ENVS = ('GITHUB_COPILOT_OAUTH_TOKEN', 'GH_COPILOT_TOKEN')
GITHUB_ENVS = ('GH_TOKEN', 'GITHUB_TOKEN')
OAUTH_ENVS = COPILOT_ENVS + GITHUB_ENVS   #: Every variable that can provide an OAuth token.

def oauth_paths():
    "Return files that can contain a GitHub OAuth token from a Copilot sign-in, in search order."
    cfg = Path(os.getenv('XDG_CONFIG_HOME') or Path.home()/'.config')
    ps  = [cfg/'rishi'/'copilot.json'] + [cfg/'github-copilot'/f for f in ('apps.json', 'hosts.json')]
    if (la := os.getenv('LOCALAPPDATA')):
        ps += [Path(la)/'github-copilot'/f for f in ('apps.json', 'hosts.json')]
    return ps + [Path.home()/'.copilot'/'config.json']

def find_oauth(o, depth=0):
    "Return the `oauth_token` in a decoded config file at nesting depth 4 or less."
    if depth > 4 or not isinstance(o, dict): return None
    if isinstance(t := o.get('oauth_token'), str) and t: return t
    return first(x for x in (find_oauth(v, depth+1) for v in o.values()) if x)

def read_oauth(p):
    "The GitHub OAuth token in config file `p`, or `None` if it has none or is not there."
    try: return find_oauth(json.loads(Path(p).read_text()))
    except Exception: return None

def _env_oauth(ks):
    "The first of environment variables `ks` that holds anything."
    return first(t for k in ks if (t := os.getenv(k)))

def copilot_oauth(token=None):
    "Return a GitHub OAuth token from the argument, `COPILOT_ENVS`, an editor sign-in, or `GITHUB_ENVS`, in that order."
    if token: return token
    if (t := _env_oauth(COPILOT_ENVS)): return t
    for p in oauth_paths():
        if (t := read_oauth(p)): return t
    if (t := _env_oauth(GITHUB_ENVS)): return t
    raise RuntimeError(
        'No GitHub OAuth token for Copilot. Set one of ' + ', '.join(OAUTH_ENVS) + ', or sign in to '
        'Copilot in an editor, or run `from rishi.copilot import copilot_login; copilot_login()`.')

In [ ]:
# every shape an editor writes, and one that holds nothing
test_eq(find_oauth({'oauth_token': 'a'}), 'a')
test_eq(find_oauth({'github.com:Iv1.b507a08c87ecfe98': {'oauth_token': 'b', 'user': 'me'}}), 'b')
test_eq(find_oauth({'hosts': {'github.com': {'oauth_token': 'c'}}}), 'c')
test_eq(find_oauth({'user': 'me'}), None)
test_eq(read_oauth('/nonexistent/apps.json'), None)

test_eq(copilot_oauth('explicit'), 'explicit')
test_eq(oauth_paths()[0].name, 'copilot.json')          # rishi's own file is read before any editor's
assert any('github-copilot' in str(p) for p in oauth_paths())

In [ ]:
# An editor sign-in comes before a general GitHub variable. That variable often holds a personal access token, and Copilot refuses one. Only a Copilot variable comes before the sign-in.
import tempfile

with tempfile.TemporaryDirectory() as d:
    f = Path(d)/'apps.json'
    f.write_text(json.dumps({'github.com:Iv1.b507a08c87ecfe98': {'oauth_token': 'gho_editor'}}))
    saved = {k: os.environ.pop(k, None) for k in OAUTH_ENVS}
    orig_paths, oauth_paths = oauth_paths, lambda: [f]
    try:
        os.environ['GITHUB_TOKEN'] = 'ghp_pat'
        test_eq(copilot_oauth(), 'gho_editor')
        os.environ['GH_COPILOT_TOKEN'] = 'gho_said_so'
        test_eq(copilot_oauth(), 'gho_said_so')
        del os.environ['GH_COPILOT_TOKEN']
        oauth_paths = lambda: []
        test_eq(copilot_oauth(), 'ghp_pat')
        del os.environ['GITHUB_TOKEN']
        test_fail(copilot_oauth, contains='No GitHub OAuth token')
    finally:
        oauth_paths = orig_paths
        for k, v in saved.items():
            os.environ.pop(k, None)
            if v is not None: os.environ[k] = v

## The Copilot token

The GitHub OAuth token is exchanged for a Copilot token that lasts about 30 minutes. `CopilotAuth` holds the token and renews it before it expires. Chats that share one `CopilotAuth` share the exchange.

`api_key=` accepts a Copilot token that you already have. rishi cannot renew that token without the GitHub OAuth token.

The same token lets rishi ask Copilot which models the account can reach:

- `copilot_catalog` returns Copilot's own description of each model.
- `copilot_models` lists the ids. It lists chat models only, unless you pass `kind=None`.
- `copilot_ctx` reads a model's prompt window from its catalogue entry.

In [ ]:
#| export
@dataclass(frozen=True)
class CopilotToken:
    "A Copilot token with its bearer value, endpoint, and expiry time."
    key: str
    base_url: str = COPILOT_API
    expires_at: float = 0.      # Unix seconds. Zero means rishi did not mint the token.

    def stale(self, skew=300):
        "Return whether the token expires within `skew` seconds. An untimed token is never stale."
        return bool(self.expires_at) and time.time() >= self.expires_at - skew

def copilot_exchange(oauth=None, *, url=COPILOT_TOKEN_URL, timeout=30):
    "Exchange a GitHub OAuth token for a `CopilotToken`."
    r = httpx.get(url, timeout=timeout,
                  headers={**copilot_hdrs(f'token {copilot_oauth(oauth)}'), 'Accept': 'application/json'})
    if r.status_code in (401, 403, 404): raise PermissionError(
        f'GitHub refused the Copilot token exchange ({r.status_code}). That account needs an active '
        'Copilot subscription, and the OAuth token has to come from an app Copilot knows: an editor '
        'sign-in or `copilot_login()`, not a personal access token. A personal access token is what '
        'the 404 means: to an app Copilot does not know, the endpoint is not there at all.')
    r.raise_for_status()
    d = r.json()
    exp = float(d.get('expires_at') or 0)
    if exp > 1e11: exp /= 1000      # Some responses use milliseconds.
    return CopilotToken(d['token'], (d.get('endpoints') or {}).get('api') or COPILOT_API, exp)

class CopilotAuth:
    "Hold a Copilot token and renew it before expiry. Chats that share one instance use the same exchange."
    def __init__(self, oauth=None, *, key=None, base_url=None, skew=300):
        store_attr()
        self.tok = CopilotToken(key, base_url or COPILOT_API) if key else None

    def token(self, force=False):
        "The live token, minted when there is none and re-minted when it is nearly out of time."
        if self.key: return self.tok    # rishi cannot renew a caller-provided token.
        if force or self.tok is None or self.tok.stale(self.skew):
            t = copilot_exchange(self.oauth)
            self.tok = replace(t, base_url=self.base_url) if self.base_url else t
        return self.tok

    def __repr__(self):
        t = self.tok
        return f'CopilotAuth({"signed in" if t else "not yet"}, {t.base_url if t else COPILOT_API})'

def copilot_catalog(auth=None, timeout=30):
    "Return every model this account can reach as `{id: entry}`, using Copilot's model entries."
    t = (auth or CopilotAuth()).token()
    r = httpx.get(f"{t.base_url.rstrip('/')}/models", headers=copilot_hdrs(f'Bearer {t.key}'), timeout=timeout)
    r.raise_for_status()
    return {m['id']: m for m in (r.json().get('data') or []) if m.get('id')}

def copilot_models(auth=None, timeout=30, kind='chat'):
    "List the model ids this account can reach, as Copilot reports them, keeping embedding and completion models too when `kind=None`."
    return [i for i, m in copilot_catalog(auth, timeout).items()
            if not kind or (m.get('capabilities') or {}).get('type') == kind]

def copilot_ctx(entry):
    "The prompt window Copilot reports for one catalogue entry, or `None` where it reports none."
    lim = ((entry or {}).get('capabilities') or {}).get('limits') or {}
    return lim.get('max_prompt_tokens') or lim.get('max_context_window_tokens') or None

In [ ]:
# The catalogue gives each model's type and window size. An embedding model answers a chat request with a 400 error that the caller cannot read. `kind` filters those models out.
_payload = {'data': [
    {'id': 'gpt-5.5', 'capabilities': {'type': 'chat', 'limits': {'max_prompt_tokens': 128000}}},
    {'id': 'claude-opus-4.7', 'capabilities': {'type': 'chat', 'limits': {'max_context_window_tokens': 264000}}},
    {'id': 'gpt-41-copilot', 'capabilities': {'type': 'completion'}},
    {'id': 'text-embedding-3-small', 'capabilities': {'type': 'embeddings'}},
    {'no_id': 'skipped'}]}

class _ModelsResp:
    status_code = 200
    def raise_for_status(self): pass
    def json(self): return _payload

orig, httpx.get = httpx.get, lambda *a, **kw: _ModelsResp()
try:
    auth = CopilotAuth(key='tid=abc')                 # a key of its own, so nothing is exchanged
    cat = copilot_catalog(auth)
    test_eq(list(cat), ['gpt-5.5', 'claude-opus-4.7', 'gpt-41-copilot', 'text-embedding-3-small'])
    test_eq(copilot_models(auth), ['gpt-5.5', 'claude-opus-4.7'])          # chat models only, by default
    test_eq(len(copilot_models(auth, kind=None)), 4)
    test_eq(copilot_models(auth, kind='embeddings'), ['text-embedding-3-small'])
    test_eq(copilot_ctx(cat['gpt-5.5']), 128000)                           # `max_prompt_tokens` first
    test_eq(copilot_ctx(cat['claude-opus-4.7']), 264000)                   # then the whole window
    test_eq(copilot_ctx(cat['gpt-41-copilot']), None)                      # no limits reported
    test_eq(copilot_ctx(None), None)
finally: httpx.get = orig

In [ ]:
now = time.time()
test_eq(CopilotToken('k').stale(), False)                     # untimed: never stale
test_eq(CopilotToken('k', expires_at=now+3600).stale(), False)
test_eq(CopilotToken('k', expires_at=now+60).stale(), True)    # inside the 5 minute skew
test_eq(CopilotToken('k', expires_at=now-1).stale(), True)

# a key you brought yourself is used as-is, and never exchanged
test_eq(CopilotAuth(key='tid=abc').token().key, 'tid=abc')
test_eq(CopilotAuth(key='k').token().base_url, COPILOT_API)
test_eq(CopilotAuth(key='k', base_url='https://api.business.githubcopilot.com').token().base_url,
        'https://api.business.githubcopilot.com')

# minting runs once, then not again until the token is nearly out of time
mints = []
def _fake_exchange(oauth=None, **kw):
    mints.append(oauth)
    return CopilotToken('tid=minted', 'https://api.individual.githubcopilot.com', time.time()+1800)

orig, copilot_exchange = copilot_exchange, _fake_exchange
try:
    a = CopilotAuth('gho_x')
    test_eq(a.token().key, 'tid=minted')
    test_eq(a.token().base_url, 'https://api.individual.githubcopilot.com')
    test_eq(len(mints), 1)                                     # the second call reused it
    a.tok = replace(a.tok, expires_at=time.time()+10)
    a.token()
    test_eq(len(mints), 2)                                     # nearly expired, so re-minted
finally: copilot_exchange = orig

In [ ]:
# The exchange answers a personal access token with a 404. That is a refusal like a 401 or a 403. The `PermissionError` tells the caller that Copilot takes no personal access token.
class _FakeResp:
    def __init__(self, code): self.status_code = code
    def raise_for_status(self): raise AssertionError('a refusal must not reach raise_for_status')

def _refused(code):
    orig, httpx.get = httpx.get, lambda *a, **kw: _FakeResp(code)
    try: copilot_exchange('gho_x')
    except PermissionError as e: return str(e)
    finally: httpx.get = orig
    assert False, f'{code} raised no PermissionError'

for code in (401, 403, 404): assert str(code) in _refused(code)

## Signing in

`copilot_login()` runs GitHub's device flow with the editor client id. It prints a code and waits while you authorize it in a browser. It saves the token where `copilot_oauth()` looks first. Pass `save=False` to skip the save. An existing editor sign-in makes this step unnecessary.

In [ ]:
#| export
DEVICE_CODE_URL  = 'https://github.com/login/device/code'   #: GitHub device-flow endpoint.
ACCESS_TOKEN_URL = 'https://github.com/login/oauth/access_token'

def save_oauth(token, path=None):
    "Write a GitHub OAuth token where `copilot_oauth` looks first, readable only by you."
    p = Path(path or oauth_paths()[0])
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps({'oauth_token': token}))
    p.chmod(0o600)
    return p

def copilot_login(client_id=COPILOT_CLIENT_ID, *, scope='read:user', save=True, printer=print, timeout=30):
    "Run GitHub's device flow: print a code, wait while you enter it, return the OAuth token."
    hdrs = {'Accept': 'application/json', 'User-Agent': USER_AGENT}
    d = httpx.post(DEVICE_CODE_URL, json={'client_id': client_id, 'scope': scope},
                   headers=hdrs, timeout=timeout).json()
    if not d.get('device_code'): raise RuntimeError(f'GitHub declined the device code request: {d}')
    printer(f"Open {d['verification_uri']} and enter {d['user_code']}")
    wait, deadline = float(d.get('interval') or 5), time.time() + float(d.get('expires_in') or 900)
    body = {'client_id': client_id, 'device_code': d['device_code'],
            'grant_type': 'urn:ietf:params:oauth:grant-type:device_code'}
    while time.time() < deadline:
        time.sleep(wait)
        r = httpx.post(ACCESS_TOKEN_URL, json=body, headers=hdrs, timeout=timeout).json()
        if (tok := r.get('access_token')):
            if save: printer(f'Saved to {save_oauth(tok)}')
            return tok
        if r.get('error') == 'slow_down': wait += float(r.get('interval') or 5)   # GitHub requests a longer polling interval.
        elif r.get('error') != 'authorization_pending':
            raise RuntimeError(f"device flow failed: {r.get('error_description') or r.get('error')}")
    raise TimeoutError('the device code expired before it was entered')

## CopilotChat

`CopilotChat` is a `RemoteChat`. Its `_kw` adds Copilot's endpoint, a live token and the required headers to each request. Tools, streaming, approval and the tool loop work as in `RemoteChat`.

Use `Chat('copilot/gpt-4.1')` or `runtime='copilot'`. With no model named, the chat uses `copilot_default`, set to `gpt-4.1`.

Only the `copilot/` prefix selects this backend automatically. Bare `gpt-...` and `claude-...` ids still go to the hosted APIs through `remote`.

In [ ]:
#| export
#: Default model used when `CopilotChat()` receives no model. `copilot_models()` lists available models.
copilot_default = 'gpt-4.1'

class CopilotChat(RemoteChat):
    "A `RemoteChat` pointed at GitHub Copilot, adding Copilot's headers and its short-lived token to each request."
    _runtime = 'copilot'
    local = False   #: Copilot runs on GitHub's machines.

    def __init__(self, model=None, *,
                 oauth_token=None,   # GitHub OAuth token. Uses the token on disk when `None`.
                 auth=None,          # `CopilotAuth` to share. Creates one when `None`.
                 integration_id=INTEGRATION_ID,
                 hdrs=None,          # Extra headers that override Copilot's headers.
                 tool_choice=None,   # Tool choice, or 'required'.
                 retries=2,
                 comp_kw=None,       # Arguments passed to `acomplete` unchanged.
                 **kw):              # Portable options from `urai.ChatOpts`.
        o = ChatOpts.create(kw.pop('opts', None), **kw)
        self.auth = auth or CopilotAuth(oauth_token, key=o.key, base_url=o.base_url or None)
        self.integration_id, self.hdrs = integration_id, dict(hdrs or {})
        super().__init__(model or copilot_default, opts=o, api_name='openai_chat',   # fastllm has no Copilot vendor. Keep Gemini ids on this route.
                         tool_choice=tool_choice, retries=retries, comp_kw=comp_kw)

    def _has_media(self):
        "Does the history carry an image or audio part? Copilot gates those behind a header."
        return any(is_media(p) for m in self.hist for p in L(m.get('content')))

    def _initiator(self):
        "`user` for a turn a person opened, `agent` once the model is round-tripping its own tool calls."
        return 'agent' if any(m.get('role') in ('assistant', 'tool') for m in self.hist) else 'user'

    def _kw(self, stream=False, **turn):
        "`RemoteChat._kw` plus Copilot's endpoint, a live token, and the headers it insists on."
        kw = super()._kw(stream, **turn)
        t  = self.auth.token()
        kw.update(api_key=t.key, base_url=t.base_url)
        kw['xtra_hdrs'] = {**copilot_hdrs(integration_id=self.integration_id, vision=self._has_media(),
                                          initiator=self._initiator()),
                           **(kw.get('xtra_hdrs') or {}), **self.hdrs}
        return kw

    async def _acomplete(self, msgs, stream=False, **turn):
        "`RemoteChat._acomplete`, with a token refresh kept off the event loop."
        await asyncio.to_thread(self.auth.token)
        return await super()._acomplete(msgs, stream, **turn)

In [ ]:
# This test supplies a token and checks the request without network access.
c = CopilotChat('copilot/gpt-4.1', api_key='tid=abc', sp='Be terse.', messages=['hello'])
test_eq(c.runtime, 'copilot')
test_eq(c.model_id, 'gpt-4.1')
test_eq(c.local, False)

kw = c._kw()
test_eq(kw['api_key'], 'tid=abc')
test_eq(kw['base_url'], COPILOT_API)
test_eq(kw['api_name'], 'openai_chat')
test_eq(kw['vendor_name'], None)
test_eq(kw['system'], 'Be terse.')
test_eq(kw['xtra_hdrs']['copilot-integration-id'], 'vscode-chat')
test_eq(kw['xtra_hdrs']['x-initiator'], 'user')
assert 'copilot-vision-request' not in kw['xtra_hdrs']
assert 'Authorization' not in kw['xtra_hdrs']               # fastllm sets it once

# Tool calls set the initiator to `agent`. Images set the vision header.
c.hist += [{'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('ls', {})]},
           {'role': 'tool', 'tool_call_id': 'x', 'name': 'ls', 'content': 'a.py'}]
test_eq(c._kw()['xtra_hdrs']['x-initiator'], 'agent')
c.hist.append({'role': 'user', 'content': [{'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,aa'}}]})
test_eq(c._kw()['xtra_hdrs']['copilot-vision-request'], 'true')

# Caller headers override rishi's headers.
test_eq(CopilotChat(api_key='k', integration_id='copilot-cli', hdrs={'editor-version': 'vim/9.1'}
                    )._kw()['xtra_hdrs']['editor-version'], 'vim/9.1')
test_eq(CopilotChat(api_key='k', integration_id='copilot-cli')._kw()['xtra_hdrs']['copilot-integration-id'],
        'copilot-cli')

# Constructor arguments reach `RemoteChat`. Gemini ids stay on the OpenAI route.
g = CopilotChat('gemini-2.5-pro', api_key='k', retries=5, tool_choice='required', comp_kw={'top_p': .5})
test_eq((g.retries, g.tool_choice, g.comp_kw), (5, 'required', {'top_p': .5}))
test_eq(g._wire_model, 'gemini-2.5-pro')
test_eq((g._kw()['api_name'], g._kw()['retries'], g._kw()['top_p']), ('openai_chat', 5, .5))

# Routing does not build a chat.
test_eq(resolve_runtime('copilot/claude-sonnet-4.5'), ('copilot', 'claude-sonnet-4.5'))
test_eq(resolve_runtime('claude-sonnet-4.5'), ('remote', 'claude-sonnet-4.5'))
test_eq(get_runtime('copilot').__name__, 'CopilotChat')   # the exported class, not this cell's

In [ ]:
# Check the request that fastllm builds from these arguments.
from fastllm.acomplete import api_registry, mk_client

kw = CopilotChat('copilot/gpt-4.1', api_key='tid=abc', sp='Be terse.', messages=['hi'])._kw()
cli, api_name, vendor = mk_client(model='gpt-4.1', vendor_name=kw['vendor_name'], api_name=kw['api_name'],
                                  api_key=kw['api_key'], base_url=kw['base_url'], xtra_hdrs=kw['xtra_hdrs'])
test_eq(cli.base_url + api_registry[api_name].endpoint, 'https://api.githubcopilot.com/chat/completions')

hdrs = dict(cli.transport.base_headers)
test_eq([k for k in hdrs if k.lower() == 'authorization'], ['Authorization'])   # exactly one
test_eq(hdrs['Authorization'], 'Bearer tid=abc')
test_eq(hdrs['copilot-integration-id'], 'vscode-chat')
test_eq(hdrs['editor-version'], EDITOR_VERSION)


## Against the real endpoint

These examples need a Copilot subscription. They are marked `eval: false`, and the test run skips them.

In [ ]:
#| eval: false
from rishi.copilot import copilot_models, copilot_login

# Run `copilot_login()` only when no editor is signed in.
print(copilot_models()[:10])

In [ ]:
#| eval: false
chat = Chat('copilot/gpt-4.1', sp='You are concise.')
r = chat('Reply with exactly: pong')
assert 'pong' in resp_text(r).lower()
print(chat.use)

In [ ]:
#| eval: false
display_stream(Chat('copilot/claude-sonnet-4.5')('Write two sentences about the monsoon.', stream=True))

In [ ]:
#| eval: false
def word_count(text: str) -> int:
    "Count the words in `text`."
    return len(text.split())

chat = Chat('copilot/gpt-4.1', tools=[word_count])   # Without `approve=`, every call runs.
print(resp_text(chat('How many words are in "the quick brown fox"? Use the tool.')))

In [ ]:
#| eval: false
# Share one auth object across chats to share one exchange.
from rishi.copilot import CopilotAuth, CopilotChat

auth = CopilotAuth()
a, b = CopilotChat('gpt-4.1', auth=auth), CopilotChat('claude-sonnet-4.5', auth=auth)
print(auth)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()